# 파일럿 (동결 전 단계) — Kaggle / Colab 겸용

`SPEC.md` §12의 1~5단계(S0 환경 점검 → S7 보고)만 돌린다. **본실험(H2)은 돌리지 않는다.**

- Kaggle: 오른쪽 설정에서 **Accelerator = GPU T4 x2**, **Internet = On**. 그다음 **Save Version → Save & Run All (Commit)**. 창을 닫아도 계속 돈다.
- 모델이 gated면 Kaggle **Add-ons → Secrets**에 `HF_TOKEN`을 넣는다 (Hugging Face에서 Gemma 4 라이선스 동의 필요).
- 결과: Output 탭의 `pilot_results.zip` (안에 `PILOT_REPORT.md`).
- Colab: 결과를 Google Drive에 저장하고, 끊기면 이 노트북을 다시 실행하면 이어서 돈다.

In [ ]:
import os, subprocess, sys, pathlib, shutil
REPO = "https://github.com/skadmlwls9455-pixel/An-Experiment-on-the-Degree-of-Flattery-Based-on-Guidelines-for-an-Agent-s-Tone-of-Voice.git"
BRANCH = "ccr-84ab013c-stns84"
MODEL_ID = "google/gemma-4-E2B-it"
ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    WORK = pathlib.Path("/kaggle/working"); OUT = WORK / "pilot"
else:  # Colab: keep results on Drive so a disconnect does not lose them
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = pathlib.Path("/content"); OUT = pathlib.Path("/content/drive/MyDrive/flattery_pilot")
OUT.mkdir(parents=True, exist_ok=True)
token = os.environ.get("HF_TOKEN")
if ON_KAGGLE and not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("kaggle" if ON_KAGGLE else "colab", "| out:", OUT, "| HF token:", bool(token))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
repo = WORK / "repo"
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(repo)], check=True)
print(subprocess.run(["git", "-C", str(repo), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate datasets
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.device_count(), "GPU(s)")

In [ ]:
# Runs S0..S7 and streams the log. Safe to re-run: finished generations are skipped.
log = open(OUT / "log.txt", "a")
p = subprocess.Popen([sys.executable, "-u", "-m", "src.run_pilot", "--out", str(OUT), "--model", MODEL_ID, "--batch", "16"],
                     cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end=""); log.write(line); log.flush()
p.wait(); log.close()
print("exit code", p.returncode)

In [ ]:
archive = shutil.make_archive(str(WORK / "pilot_results"), "zip", OUT)
print(archive)
rep = OUT / "PILOT_REPORT.md"
print(rep.read_text() if rep.exists() else "no report (see log.txt)")